1.LOAD DATASET

In [1]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

PLATFORM = "tiktok"
SEEDS = [42, 123, 2024, 7, 99]
DATA_DIR = "../data_preprocess/processed_data/ml_data/"

results = []

for seed in SEEDS:
    train_df = pd.read_csv(f"{DATA_DIR}{PLATFORM}_train_seed{seed}.csv")
    test_df  = pd.read_csv(f"{DATA_DIR}{PLATFORM}_test_seed{seed}.csv")

    neg, pos = train_df["popularity"].value_counts()[0], train_df["popularity"].value_counts()[1]
    ratio = neg / pos

    X_train = train_df.drop(columns=["post_id", "user_id", "popularity"])
    y_train = train_df["popularity"]
    X_test  = test_df.drop(columns=["post_id", "user_id", "popularity"])
    y_test  = test_df["popularity"]

    model = XGBClassifier(
        objective="binary:logistic", eval_metric="logloss",
        scale_pos_weight=ratio, n_estimators=500, learning_rate=0.02,
        max_depth=6, min_child_weight=1, gamma=0.1,
        subsample=0.8, colsample_bytree=0.5,
        random_state=seed, n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "seed": seed,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
    })
    print(f"[seed {seed}] done | F1={results[-1]['f1']:.4f} | ROC-AUC={results[-1]['roc_auc']:.4f}")

results_df = pd.DataFrame(results)
print("\n" + "="*60)
print(f"{PLATFORM.upper()} BASELINE — MEAN ± STD ACROSS {len(SEEDS)} SEEDS")
print("="*60)
print(results_df.set_index("seed"))
print("\nMean ± Std:")
summary = results_df.drop(columns="seed").agg(["mean", "std"])
print(summary)

[seed 42] done | F1=0.6065 | ROC-AUC=0.8705
[seed 123] done | F1=0.6284 | ROC-AUC=0.8917
[seed 2024] done | F1=0.6198 | ROC-AUC=0.8830
[seed 7] done | F1=0.6095 | ROC-AUC=0.8799
[seed 99] done | F1=0.5885 | ROC-AUC=0.8707

TIKTOK BASELINE — MEAN ± STD ACROSS 5 SEEDS
      accuracy  precision    recall        f1   roc_auc
seed                                                   
42    0.801934   0.503467  0.762605  0.606516  0.870454
123   0.814971   0.525424  0.781513  0.628378  0.891700
2024  0.806560   0.510899  0.787815  0.619835  0.882979
7     0.810345   0.518409  0.739496  0.609524  0.879910
99    0.794786   0.491549  0.733193  0.588533  0.870706

Mean ± Std:
      accuracy  precision    recall        f1  roc_auc
mean  0.805719   0.509950  0.760924  0.610557  0.87915
std   0.007771   0.013158  0.024382  0.015051  0.00894


In [2]:
results_df.insert(0, "model", "baseline_metadata")
results_df.insert(0, "platform", PLATFORM)

import os
os.makedirs("../results", exist_ok=True)
results_df.to_csv(f"../results/{PLATFORM}_baseline_metadata.csv", index=False)
print(f"Saved: ../results/{PLATFORM}_baseline_metadata.csv")

Saved: ../results/tiktok_baseline_metadata.csv
